In [47]:
import gensim
from owlready2 import *
import polars as pl
import numpy as np

In [2]:
word2vec_embedding_file = "../embeddings/ontology.embeddings"
onto_file = "../data/efo_otar_slim.owl"

model = gensim.models.Word2Vec.load(word2vec_embedding_file)
onto = get_ontology(onto_file).load()
classes = list(onto.classes())
c = classes[0]

iri_v = model.wv.get_vector(c.iri)

In [12]:
all_classes_iri = [c.iri for c in classes]

print(f"Number of class iri: {len(all_classes_iri)}")

Number of class iri: 57096


In [9]:
# read disease OT file 

disease_df = pl.read_parquet("../data/disease/disease.parquet")
disease_df.head(1)


id,code,name,description,dbXRefs,parents,exactSynonyms,relatedSynonyms,narrowSynonyms,broadSynonyms,obsoleteTerms,obsoleteXRefs,children,ancestors,therapeuticAreas,descendants,ontology,synonyms
str,str,str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],struct[3],struct[4]
"""DOID_0050890""","""http://purl.obolibrary.org/obo…","""synucleinopathy""","""A neurodegenerative disease th…","[""MEDGEN:1682194"", ""MESH:D000080874"", … ""UMLS:C5191670""]","[""EFO_0005772"", ""MONDO_0021179""]","[""alpha Synucleinopathies"", ""synucleinopathy""]","[""alpha synucleinopathies"", ""synucleinopathies""]",[],[],[],[],"[""EFO_0006792"", ""EFO_1001050""]","[""EFO_0005772"", ""MONDO_0021179"", … ""OTAR_0000020""]","[""EFO_0000618"", ""OTAR_0000020""]","[""EFO_0006792"", ""EFO_1001050"", … ""MONDO_0014835""]","{false,false,{""http://purl.obolibrary.org/obo/DOID_0050890"",""DOID_0050890""}}","{[""alpha Synucleinopathies"", ""synucleinopathy""],[""alpha synucleinopathies"", ""synucleinopathies""],[],[]}"


In [ ]:
disease_iri_codes = disease_df['code'].to_list()

# are all OT disease codes in EFO_slim iri? --> YES

print(f"N. of OT diseases: {len(set(disease_iri_codes))}")
print(f"N. of intersection: {len(set(disease_iri_codes) & set(all_classes_iri))}")


N. of OT diseases: 47030
N. of intersection: 47030


In [83]:

def get_embedding(iri):
    return model.wv.get_vector(iri).astype(np.float32).tolist()

embeddings = pl.DataFrame({'iri': disease_iri_codes}).with_columns(
    pl.col("iri")
    .map_elements(
        get_embedding,
        return_dtype=pl.List(pl.Float32)
    )
    .alias("embeddings")
)


In [84]:
embeddings.head(2)

iri,embeddings
str,list[f32]
"""http://purl.obolibrary.org/obo…","[0.000494, -0.159378, … 0.179417]"
"""http://purl.obolibrary.org/obo…","[0.047898, -0.144157, … 0.198342]"


In [85]:
embeddings.write_parquet("../output/embeddings.parquet", compression="brotli")

In [82]:
embeddings.height

57096